# Rolling, Expanding & Window Functions

## Definition
Window functions compute statistics over a **sliding or expanding window** of rows.
Essential for time-series analysis and temporal feature engineering.

### Types
| Type | Method | Description |
|------|--------|-------------|
| **Rolling** | `df.rolling(n)` | Fixed-size sliding window |
| **Expanding** | `df.expanding()` | Grows from start |
| **EWM** | `df.ewm(span)` | Exponential weighted (more weight to recent) |

### Rolling Parameters
| Param | Description |
|-------|-------------|
| `window` | Window size (int or offset like '7D') |
| `min_periods` | Minimum non-null values required |
| `center` | Center the window |
| `closed` | Which end to close ('left', 'right', 'both', 'neither') |

In [ ]:
import pandas as pd
import numpy as np

# Daily stock price simulation
rng = np.random.default_rng(42)
dates = pd.date_range('2023-01-01', periods=60, freq='B')  # business days
prices = pd.Series(100 + np.cumsum(rng.normal(0, 1.5, 60)), index=dates, name='price')
df = prices.to_frame()
print(df.head(10))

In [ ]:
# Rolling mean (moving average)
df['ma_7']  = df['price'].rolling(window=7).mean()   # 7-day MA
df['ma_20'] = df['price'].rolling(window=20).mean()  # 20-day MA

print('Rolling means (first 25 rows):')
print(df.head(25))

In [ ]:
# Rolling with various stats
df['roll_std']  = df['price'].rolling(7).std()
df['roll_min']  = df['price'].rolling(7).min()
df['roll_max']  = df['price'].rolling(7).max()
df['roll_range'] = df['roll_max'] - df['roll_min']

print('Rolling stats:')
print(df[['price', 'roll_std', 'roll_min', 'roll_max']].head(15).round(2))

In [ ]:
# min_periods — allow partial windows at start
df['ma7_minp'] = df['price'].rolling(window=7, min_periods=1).mean()
print('First 7 rows with min_periods=1:')
print(df[['price', 'ma_7', 'ma7_minp']].head(8).round(2))

In [ ]:
# Expanding window — cumulative stats from start
df['expanding_mean'] = df['price'].expanding().mean()
df['expanding_max']  = df['price'].expanding().max()
df['expanding_std']  = df['price'].expanding().std()

print('Expanding stats:')
print(df[['price', 'expanding_mean', 'expanding_max']].head(10).round(2))

In [ ]:
# Exponential weighted mean (EWM) — more weight to recent values
df['ewm_12'] = df['price'].ewm(span=12, adjust=False).mean()
df['ewm_26'] = df['price'].ewm(span=26, adjust=False).mean()

# MACD-like signal
df['macd'] = df['ewm_12'] - df['ewm_26']
print('EWM and MACD:')
print(df[['price', 'ewm_12', 'ewm_26', 'macd']].tail(10).round(3))

In [ ]:
# Lag features using shift() — very important for time-series ML
df['lag_1'] = df['price'].shift(1)   # yesterday's price
df['lag_7'] = df['price'].shift(7)   # 7 days ago
df['pct_change_1'] = df['price'].pct_change(1)  # daily return
df['pct_change_7'] = df['price'].pct_change(7)  # weekly return

print('Lag and pct_change features:')
print(df[['price', 'lag_1', 'lag_7', 'pct_change_1']].head(12).round(4))

In [ ]:
# Rolling with groupby (group-wise rolling)
df_groups = pd.DataFrame({
    'store': ['A', 'A', 'A', 'A', 'B', 'B', 'B', 'B'],
    'date': pd.date_range('2023-01-01', periods=4).tolist() * 2,
    'sales': [100, 120, 110, 130, 80, 90, 85, 95]
})
df_groups['rolling_avg'] = (df_groups.groupby('store')['sales']
                            .transform(lambda x: x.rolling(2).mean()))
print('Per-store rolling average:')
print(df_groups)